In [1]:
import pandas as pd
import re
import os

def update_and_parse_datasets():
    print("🚀 데이터 덮어쓰기 및 파싱 작업을 시작합니다...\n")

    # ==========================================
    # 1. Problems 데이터 덮어쓰기 및 파싱
    # ==========================================
    print("1️⃣ Problems 데이터 병합 및 시간/메모리 파싱 중...")
    try:
        # 파일 불러오기
        base_prob = pd.read_csv('filtered_concat_translated_problems_polished.csv')
        ssajun_prob = pd.read_csv('ssajun_problems.csv')
        
        # 'id'를 인덱스로 설정하여 ssajun_problems의 내용으로 원본 덮어쓰기
        base_prob.set_index('id', inplace=True)
        ssajun_prob.set_index('id', inplace=True)
        
        base_prob.update(ssajun_prob)
        base_prob.reset_index(inplace=True)
        
        # --- 시간 파싱 함수 ---
        def parse_time(val):
            if pd.isna(val): return 3.0
            s = str(val).lower()
            # 정수 또는 소수점 숫자만 추출
            match = re.search(r'(\d+(?:\.\d+)?)', s)
            if not match: return 3.0
            
            num = float(match.group(1))
            if 'mili' in s or 'ms' in s:
                num /= 1000.0
                
            # 숫자가 0.0001 이하면 3으로 세팅
            if num <= 0.0001:
                return 3.0
            return num

        # --- 메모리 파싱 함수 ---
        def parse_memory(val):
            if pd.isna(val): return 256.0
            s = str(val).lower()
            match = re.search(r'(\d+(?:\.\d+)?)', s)
            if not match: return 256.0
            
            num = float(match.group(1))
            if 'giga' in s or 'gb' in s:
                num *= 1024.0
            elif 'kilo' in s or 'kb' in s:
                num /= 1024.0
            elif 'mega' not in s and 'mb' not in s:
                # giga, kilo, mega, mb 모두 없으면 순수 byte로 간주
                num /= (1024.0 * 1024.0)
            
            # 숫자가 0.0001 이하면 256으로 세팅
            if num <= 0.0001:
                return 256.0
            return num

        # 새로운 파싱 열 적용
        base_prob['time_limit_parsed(s)'] = base_prob['time_limit'].apply(parse_time)
        base_prob['memory_limit_parsed(mb)'] = base_prob['memory_limit'].apply(parse_memory)
        
        # 결과 저장
        base_prob.to_csv('overwritten_problems.csv', index=False, encoding='utf-8-sig')
        print("✅ 'overwritten_problems.csv' 저장 완료 (파싱 열 추가됨).")
        
    except Exception as e:
        print(f"❌ Problems 데이터 처리 중 오류 발생: {e}")

    # ==========================================
    # 2. Testcases 데이터 덮어쓰기 (배치 처리)
    # ==========================================
    print("\n2️⃣ Testcases 데이터 청크(배치 10000) 처리 중...")
    try:
        # ssajun_testcases는 크기가 상대적으로 작으므로 한 번에 메모리에 로드
        ssajun_tc = pd.read_csv('problem_testcases.csv')
        # 다중 조건(problem_id, testcase_order)을 인덱스로 설정
        ssajun_tc.set_index(['problem_id', 'testcase_order'], inplace=True)
        
        chunksize = 10000
        output_file = 'overwritten_testcases.csv'
        
        # 기존에 쓰다 만 파일이 있다면 삭제
        if os.path.exists(output_file):
            os.remove(output_file)
            
        chunk_count = 0
        for chunk in pd.read_csv('problem_testcases.csv', chunksize=chunksize):
            chunk.set_index(['problem_id', 'testcase_order'], inplace=True)
            
            # ssajun_tc의 내용으로 겹치는 행/열 덮어쓰기
            chunk.update(ssajun_tc)
            chunk.reset_index(inplace=True)
            
            # 첫 번째 청크일 때만 헤더(열 이름)를 넣고 새로 쓰기('w'), 이후엔 이어 붙이기('a')
            mode = 'w' if chunk_count == 0 else 'a'
            header = (chunk_count == 0)
            
            chunk.to_csv(output_file, mode=mode, header=header, index=False, encoding='utf-8-sig')
            
            chunk_count += 1
            print(f"   - {chunk_count * chunksize}번째 행까지 스캔 및 덮어쓰기 완료...")
            
        print("✅ 'overwritten_testcases.csv' 저장 완료.")
        
    except Exception as e:
        print(f"❌ Testcases 데이터 처리 중 오류 발생: {e}")

# 함수 실행
update_and_parse_datasets()

🚀 데이터 덮어쓰기 및 파싱 작업을 시작합니다...

1️⃣ Problems 데이터 병합 및 시간/메모리 파싱 중...
✅ 'overwritten_problems.csv' 저장 완료 (파싱 열 추가됨).

2️⃣ Testcases 데이터 청크(배치 10000) 처리 중...
   - 10000번째 행까지 스캔 및 덮어쓰기 완료...
   - 20000번째 행까지 스캔 및 덮어쓰기 완료...
   - 30000번째 행까지 스캔 및 덮어쓰기 완료...
   - 40000번째 행까지 스캔 및 덮어쓰기 완료...
   - 50000번째 행까지 스캔 및 덮어쓰기 완료...
   - 60000번째 행까지 스캔 및 덮어쓰기 완료...
   - 70000번째 행까지 스캔 및 덮어쓰기 완료...
   - 80000번째 행까지 스캔 및 덮어쓰기 완료...
   - 90000번째 행까지 스캔 및 덮어쓰기 완료...
   - 100000번째 행까지 스캔 및 덮어쓰기 완료...
   - 110000번째 행까지 스캔 및 덮어쓰기 완료...
   - 120000번째 행까지 스캔 및 덮어쓰기 완료...
   - 130000번째 행까지 스캔 및 덮어쓰기 완료...
   - 140000번째 행까지 스캔 및 덮어쓰기 완료...
   - 150000번째 행까지 스캔 및 덮어쓰기 완료...
   - 160000번째 행까지 스캔 및 덮어쓰기 완료...
   - 170000번째 행까지 스캔 및 덮어쓰기 완료...
   - 180000번째 행까지 스캔 및 덮어쓰기 완료...
   - 190000번째 행까지 스캔 및 덮어쓰기 완료...
   - 200000번째 행까지 스캔 및 덮어쓰기 완료...
   - 210000번째 행까지 스캔 및 덮어쓰기 완료...
   - 220000번째 행까지 스캔 및 덮어쓰기 완료...
   - 230000번째 행까지 스캔 및 덮어쓰기 완료...
   - 240000번째 행까지 스캔 및 덮어쓰기 완료...
   - 250000번째 행까지 스캔 및 덮어쓰기 완료...
   - 

In [3]:
import pandas as pd
import re
import os

def update_and_parse_datasets():
    print("🚀 데이터 정제, 덮어쓰기 및 파싱 작업을 시작합니다...\n")

    # ==========================================
    # 1. Problems 데이터 정제, 덮어쓰기 및 파싱
    # ==========================================
    print("1️⃣ Problems 데이터 정제 및 시간/메모리 파싱 중...")
    try:
        # 파일 불러오기
        base_prob = pd.read_csv('filtered_concat_translated_problems_polished.csv')
        ssajun_prob = pd.read_csv('ssajun_problems.csv')
        
        # [수정 1] id가 비어있는(NaN) 행 삭제
        base_prob = base_prob.dropna(subset=['id'])
        
        # [수정 2] 'Unnamed: 숫자' 형태의 열 모두 제거
        unnamed_cols = [col for col in base_prob.columns if str(col).startswith('Unnamed:')]
        if unnamed_cols:
            base_prob = base_prob.drop(columns=unnamed_cols)
            print(f"   - 🗑️ 불필요한 열 제거 완료: {unnamed_cols}")

        # 'id'를 인덱스로 설정하여 ssajun_problems의 내용으로 원본 덮어쓰기
        base_prob.set_index('id', inplace=True)
        ssajun_prob.set_index('id', inplace=True)
        
        base_prob.update(ssajun_prob)
        base_prob.reset_index(inplace=True)
        
        # --- 시간 파싱 함수 (초 -> 밀리초 변환 로직 적용) ---
        def parse_time_ms(val):
            # 빈 값이면 기본값 3초 -> 3000ms
            if pd.isna(val): return 3000.0
            
            s = str(val).lower()
            match = re.search(r'(\d+(?:\.\d+)?)', s)
            
            # 숫자가 없으면 기본값 3000ms
            if not match: return 3000.0
            
            num = float(match.group(1))
            
            # mili나 ms가 "없다면" 초(s) 단위이므로 1000을 곱해서 ms로 만들어줌.
            # (만약 mili나 ms가 있다면 이미 밀리초 단위이므로 숫자를 그대로 둠)
            if 'mili' not in s and 'ms' not in s:
                num *= 1000.0
                
            # 최종 계산된 밀리초 값이 너무 작으면(예: 0.1ms 이하) 에러/오기입으로 간주하고 3000ms 세팅
            if num <= 0.1:
                return 3000.0
            return num

        # --- 메모리 파싱 함수 (기존과 동일) ---
        def parse_memory(val):
            if pd.isna(val): return 256.0
            s = str(val).lower()
            match = re.search(r'(\d+(?:\.\d+)?)', s)
            if not match: return 256.0
            
            num = float(match.group(1))
            if 'giga' in s or 'gb' in s:
                num *= 1024.0
            elif 'kilo' in s or 'kb' in s:
                num /= 1024.0
            elif 'mega' not in s and 'mb' not in s:
                # byte인 경우
                num /= (1024.0 * 1024.0)
            
            if num <= 0.0001:
                return 256.0
            return num

        # [수정 3] 파싱 결과를 계산한 뒤, 기존 열의 '바로 오른쪽'에 삽입(insert)
        # 1) 만약 이전 버전의 (s) 열이 있다면 지워주고, 새로 만들 (ms) 열도 미리 덮어쓰기 방지로 지워줌
        if 'time_limit_parsed(s)' in base_prob.columns:
            base_prob = base_prob.drop(columns=['time_limit_parsed(s)'])
        if 'time_limit_parsed(ms)' in base_prob.columns:
            base_prob = base_prob.drop(columns=['time_limit_parsed(ms)'])
        if 'memory_limit_parsed(mb)' in base_prob.columns:
            base_prob = base_prob.drop(columns=['memory_limit_parsed(mb)'])

        # 2) 파싱 데이터 생성
        parsed_time = base_prob['time_limit'].apply(parse_time_ms)
        parsed_memory = base_prob['memory_limit'].apply(parse_memory)
        
        # 3) 기존 열의 인덱스를 찾아 바로 그 다음(+1) 위치에 삽입
        idx_time = base_prob.columns.get_loc('time_limit')
        base_prob.insert(idx_time + 1, 'time_limit_parsed(ms)', parsed_time) # 이름 변경됨!
        
        idx_mem = base_prob.columns.get_loc('memory_limit')
        base_prob.insert(idx_mem + 1, 'memory_limit_parsed(mb)', parsed_memory)
        
        # 결과 저장
        base_prob.to_csv('overwritten_problems.csv', index=False, encoding='utf-8-sig')
        print("✅ 'overwritten_problems.csv' 저장 완료 (시간 단위 ms 적용 완료).")
        
    except Exception as e:
        print(f"❌ Problems 데이터 처리 중 오류 발생: {e}")

    # ==========================================
    # 2. Testcases 데이터 덮어쓰기 (배치 처리)
    # ==========================================
    print("\n2️⃣ Testcases 데이터 청크(배치 10000) 처리 중...")
    try:
        ssajun_tc = pd.read_csv('ssajun_testcases.csv')
        ssajun_tc.set_index(['problem_id', 'testcase_order'], inplace=True)
        
        chunksize = 10000
        output_file = 'overwritten_testcases.csv'
        
        if os.path.exists(output_file):
            os.remove(output_file)
            
        chunk_count = 0
        for chunk in pd.read_csv('problem_testcases.csv', chunksize=chunksize):
            chunk.set_index(['problem_id', 'testcase_order'], inplace=True)
            
            chunk.update(ssajun_tc)
            chunk.reset_index(inplace=True)
            
            mode = 'w' if chunk_count == 0 else 'a'
            header = (chunk_count == 0)
            
            chunk.to_csv(output_file, mode=mode, header=header, index=False, encoding='utf-8-sig')
            
            chunk_count += 1
            print(f"   - {chunk_count * chunksize}번째 행까지 스캔 및 덮어쓰기 완료...")
            
        print("✅ 'overwritten_testcases.csv' 저장 완료.")
        
    except Exception as e:
        print(f"❌ Testcases 데이터 처리 중 오류 발생: {e}")

# 함수 실행
update_and_parse_datasets()

🚀 데이터 정제, 덮어쓰기 및 파싱 작업을 시작합니다...

1️⃣ Problems 데이터 정제 및 시간/메모리 파싱 중...
   - 🗑️ 불필요한 열 제거 완료: ['Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52', 'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56', 'Unnamed: 57', 'Unnamed: 58', 'Unnamed: 59', 'Unnamed: 60', 'Unnamed: 61', 'Unnamed: 62', 'Unnamed: 63', 'Unnamed: 64', 'Unnamed: 65', 'Unnamed: 66', 'Unnamed: 67', 'Unnamed: 68', 'Unnamed: 69', 'Unnamed: 70', 'Unnamed: 71', 'Unnamed: 72', 'Unnamed: 73', 'Unnamed: 74', 'Unnamed: 75', 'Unnamed: 76', 'Unnamed: 77', 'Unnamed: 78', 'Unnam

KeyboardInterrupt: 

In [2]:
import pandas as pd
import csv
import os
import sys

# ==========================================
# 💡 [근원적 해결책] CSV 필드 크기 제한 해제
# 파이썬 csv 모듈의 기본 셀 크기 제한(131,072자)을 시스템 최대치로 강제 확장합니다.
# ==========================================
maxInt = sys.maxsize
while True:
    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)
# ==========================================

def robust_testcase_pipeline():
    print("🚀 완벽한 CSV 파싱을 통한 테스트케이스 정제 및 필터링 작업을 시작합니다...\n")

    # ==========================================
    # 0. ssajun_testcases.csv 데이터를 딕셔너리로 준비 (빠른 덮어쓰기 용도)
    # ==========================================
    ssajun_dict = {}
    try:
        ssajun_df = pd.read_csv('ssajun_testcases.csv')
        for _, row in ssajun_df.iterrows():
            pid = str(int(float(row['problem_id'])))
            tc_order = str(int(float(row['testcase_order'])))
            ssajun_dict[(pid, tc_order)] = {
                'input': str(row['input']),
                'output': str(row['output'])
            }
        print(f"✅ 'ssajun_testcases.csv' 로드 완료 ({len(ssajun_dict)}개 덮어쓰기 대기)")
    except Exception as e:
        print(f"⚠️ 'ssajun_testcases.csv' 로드 실패: {e}")

    # ==========================================
    # 1. problem_testcases.csv 정제 (열 변경, 병합, is_hidden 처리)
    # ==========================================
    print("\n⏳ 1단계: 'problem_testcases.csv' 정제 및 덮어쓰기 진행 중...")
    in_file = 'problem_testcases.csv'
    out_file = 'overwritten_testcases.csv'

    try:
        with open(in_file, mode='r', encoding='utf-8-sig', newline='') as f_in, \
             open(out_file, mode='w', encoding='utf-8-sig', newline='') as f_out:
            
            # escapechar='\\' 로 문자열 내부의 특수 따옴표(\") 완벽 방어
            reader = csv.reader(f_in, escapechar='\\')
            writer = csv.writer(f_out, quoting=csv.QUOTE_MINIMAL, escapechar='\\')

            header = next(reader)
            
            # 기존 열 인덱스 파악
            pid_idx = header.index('problem_id')
            tc_idx = header.index('testcase_order')
            input_idx = header.index('input')
            output_idx = header.index('output')
            val_idx = header.index('validation') if 'validation' in header else -1

            # 새로운 헤더 구성 (이름 변경 및 제외)
            new_header = []
            for i, col in enumerate(header):
                if i == input_idx:
                    new_header.append('input_text')
                elif i == output_idx:
                    new_header.append('expected_output')
                elif i == val_idx:
                    continue  # validation 열은 제외
                else:
                    new_header.append(col)
            
            # 새 열 추가
            new_header.append('is_hidden')
            new_header.append('is_deleted')
            writer.writerow(new_header)
            
            total_rows = 0
            for row in reader:
                if len(row) < len(header):
                    continue  # 완전히 깨진 찌꺼기 행 건너뛰기
                    
                # 안전한 정수 변환 (problem_id, testcase_order)
                try:
                    pid_val = str(int(float(row[pid_idx].strip())))
                    tc_val = int(float(row[tc_idx].strip()))
                    tc_str = str(tc_val)
                except ValueError:
                    continue # ID나 순서가 숫자가 아니면 스킵

                # ssajun_testcases.csv 데이터로 덮어쓰기
                curr_input = row[input_idx]
                curr_output = row[output_idx]
                if (pid_val, tc_str) in ssajun_dict:
                    curr_input = ssajun_dict[(pid_val, tc_str)]['input']
                    curr_output = ssajun_dict[(pid_val, tc_str)]['output']

                # 새 행 조립
                new_row = []
                for i, val in enumerate(row):
                    if i == input_idx:
                        new_row.append(curr_input)
                    elif i == output_idx:
                        new_row.append(curr_output)
                    elif i == val_idx:
                        continue
                    else:
                        new_row.append(val)
                        
                # is_hidden 판별 (1, 2번만 False)
                is_hidden = False if tc_val in [1, 2] else True
                new_row.append(is_hidden)
                
                # is_deleted 빈 값 설정
                new_row.append('')
                
                writer.writerow(new_row)
                total_rows += 1
                
                if total_rows % 10000 == 0:
                    print(f"   - {total_rows:,}행 변환 및 저장 완료...")
                    
        print(f"✅ '{out_file}' 저장 완료! (총 {total_rows:,}행)\n")
        
    except Exception as e:
        print(f"❌ 1단계 처리 중 오류 발생: {e}")
        return

    # ==========================================
    # 2. overwritten_problems.csv 기준으로 필터링
    # ==========================================
    print("⏳ 2단계: 'overwritten_problems.csv' 문제 ID 기준으로 필터링 중...")
    try:
        prob_df = pd.read_csv('overwritten_problems.csv')
        # 매칭을 위해 안전하게 문자열 정수로 변환하여 Set 생성
        valid_pids = set()
        for val in prob_df['id']:
            if pd.notna(val):
                valid_pids.add(str(int(float(val))))
        print(f"✅ 기준 문제 ID {len(valid_pids)}개 로드 완료.")
    except Exception as e:
        print(f"❌ overwritten_problems.csv 로드 실패: {e}")
        return

    filtered_out_file = 'overwritten_testcases (filtered).csv'
    
    try:
        with open(out_file, mode='r', encoding='utf-8-sig', newline='') as f_in, \
             open(filtered_out_file, mode='w', encoding='utf-8-sig', newline='') as f_out:
            
            reader = csv.reader(f_in, escapechar='\\')
            writer = csv.writer(f_out, quoting=csv.QUOTE_MINIMAL, escapechar='\\')
            
            header = next(reader)
            writer.writerow(header)
            pid_idx = header.index('problem_id')
            
            kept_rows = 0
            for row in reader:
                if len(row) > pid_idx:
                    try:
                        pid_val = str(int(float(row[pid_idx].strip())))
                        # ID가 valid_pids 집합에 포함되어 있을 때만 저장
                        if pid_val in valid_pids:
                            writer.writerow(row)
                            kept_rows += 1
                    except ValueError:
                        continue
                        
        print(f"✅ '{filtered_out_file}' 저장 완료! (최종 추출: {kept_rows:,}행)\n")
        print("🎉 모든 테스트케이스 정제 및 필터링 파이프라인이 완벽하게 종료되었습니다!")
        
    except Exception as e:
        print(f"❌ 2단계 필터링 중 오류 발생: {e}")

# 함수 실행
robust_testcase_pipeline()

🚀 완벽한 CSV 파싱을 통한 테스트케이스 정제 및 필터링 작업을 시작합니다...

✅ 'ssajun_testcases.csv' 로드 완료 (871개 덮어쓰기 대기)

⏳ 1단계: 'problem_testcases.csv' 정제 및 덮어쓰기 진행 중...
   - 10,000행 변환 및 저장 완료...
   - 20,000행 변환 및 저장 완료...
   - 30,000행 변환 및 저장 완료...
   - 40,000행 변환 및 저장 완료...
   - 50,000행 변환 및 저장 완료...
   - 60,000행 변환 및 저장 완료...
   - 70,000행 변환 및 저장 완료...
   - 80,000행 변환 및 저장 완료...
   - 90,000행 변환 및 저장 완료...
   - 100,000행 변환 및 저장 완료...
   - 110,000행 변환 및 저장 완료...
   - 120,000행 변환 및 저장 완료...
   - 130,000행 변환 및 저장 완료...
   - 140,000행 변환 및 저장 완료...
   - 150,000행 변환 및 저장 완료...
   - 160,000행 변환 및 저장 완료...
   - 170,000행 변환 및 저장 완료...
   - 180,000행 변환 및 저장 완료...
   - 190,000행 변환 및 저장 완료...
   - 200,000행 변환 및 저장 완료...
   - 210,000행 변환 및 저장 완료...
   - 220,000행 변환 및 저장 완료...
   - 230,000행 변환 및 저장 완료...
   - 240,000행 변환 및 저장 완료...
   - 250,000행 변환 및 저장 완료...
   - 260,000행 변환 및 저장 완료...
   - 270,000행 변환 및 저장 완료...
   - 280,000행 변환 및 저장 완료...
   - 290,000행 변환 및 저장 완료...
   - 300,000행 변환 및 저장 완료...
   - 310,000행 변환 및 저장 완료..

In [4]:
import pandas as pd

def inspect_filtered_testcases():
    file_name = 'overwritten_testcases (filtered).csv'
    print(f"🔍 최종 결과물 '{file_name}'의 후반부 상태를 점검합니다...\n")
    
    try:
        # 이전에 기록할 때 사용했던 안전한 이스케이프 문자를 동일하게 적용하여 읽어옵니다.
        print("📂 데이터를 불러오는 중입니다. (안전 파싱 모드 적용)...")
        df = pd.read_csv(file_name, 
                         escapechar='\\', 
                         engine='python', 
                         on_bad_lines='skip')
        
        total_rows = len(df)
        print(f"✅ 파일 로드 완료! (총 행 개수: {total_rows:,}개)\n")
        
        if total_rows == 0:
            print("❌ 데이터가 비어있습니다. 필터링 과정에서 문제가 있었는지 확인이 필요합니다.")
            return
            
        if total_rows <= 10000:
            print("⚠️ 전체 행이 10,000개 이하입니다. 가능한 전체 범위 내에서 샘플링합니다.")
            target_df = df
        else:
            # 10000번째 행 이후의 데이터만 타겟으로 설정 (인덱스 기준)
            target_df = df.iloc[10000:]
            
        # 최대 100개 샘플링 (고정된 시드값을 주어 매번 같은 100개를 볼 수 있게 설정)
        sample_size = min(100, len(target_df))
        sampled_df = target_df.sample(n=sample_size, random_state=42)
        
        print(f"📊 10,000행 이후의 데이터 중 {sample_size}개를 랜덤 추출한 결과입니다:\n")
        
        try:
            # Jupyter 환경에서 깔끔한 표 형태로 출력
            from IPython.display import display
            display(sampled_df)
        except ImportError:
            # 일반 터미널 환경일 경우
            print(sampled_df.to_string())
            
        # 간단한 양식 훼손 자가진단 (problem_id 열이 숫자로 변환 가능한지 체크)
        invalid_pids = pd.to_numeric(sampled_df['problem_id'], errors='coerce').isna().sum()
        if invalid_pids > 0:
            print(f"\n⚠️ 경고: 추출된 샘플 중 problem_id가 숫자가 아닌 행이 {invalid_pids}개 발견되었습니다. 양식이 여전히 밀렸을 수 있습니다.")
        else:
            print("\n🎉 진단: 추출된 샘플의 problem_id가 모두 정상적인 숫자입니다! 열 밀림 현상이 완벽하게 해결된 것으로 보입니다.")
            
    except Exception as e:
        print(f"❌ 파일을 읽거나 처리하는 중 오류가 발생했습니다: {e}")

# 함수 실행
inspect_filtered_testcases()

🔍 최종 결과물 'result_testcases.csv'의 후반부 상태를 점검합니다...

📂 데이터를 불러오는 중입니다. (안전 파싱 모드 적용)...
✅ 파일 로드 완료! (총 행 개수: 1,048,575개)

📊 10,000행 이후의 데이터 중 100개를 랜덤 추출한 결과입니다:



,problem_id,testcase_order,input_text,expected_output,is_hidden,is_deleted,Unnamed: 6,Unnamed: 7,Unnamed: 8,is_hidden.1
178906,27665,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
626255,1959 1960,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
658954,1667 4167 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
678520,289 290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
299672,705 576 21 83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
...,...,...,...,...,...,...,...,...,...,...
429517,332627142 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
678774,7 1969 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
784936,3420262,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
489376,88236471 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True



⚠️ 경고: 추출된 샘플 중 problem_id가 숫자가 아닌 행이 71개 발견되었습니다. 양식이 여전히 밀렸을 수 있습니다.
